In [2]:
!pip install ucimlrepo


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Module 1


In [3]:
from ucimlrepo import fetch_ucirepo

ai4i = fetch_ucirepo(id=601)
X = ai4i.data.features
y = ai4i.data.targets["Machine failure"]

print(X.shape)
print(X.columns.tolist())
print(ai4i.data.targets.columns.tolist())
print(y.value_counts())

(10000, 6)
['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear']
['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
Machine failure
0    9661
1     339
Name: count, dtype: int64


La data est Déséquilibrer

In [4]:
import pandas as pd

df = X.copy()
df["Machine failure"] = y

In [5]:
print(df.shape)       # nombre de lignes et colonnes
df.head(10)           # les 10 premières lignes


(10000, 7)


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0
5,M,298.1,308.6,1425,41.9,11,0
6,L,298.1,308.6,1558,42.4,14,0
7,L,298.1,308.6,1527,40.2,16,0
8,M,298.3,308.7,1667,28.6,18,0
9,M,298.5,309.0,1741,28.0,21,0


Feature	Signification	Unité
Type	Qualité du produit fabriqué : L (basse), M (moyenne), H (haute)	catégorie
Air temperature	Température de l’air ambiant autour de la machine	Kelvin (environ 300 K, soit 27 °C)
Process temperature	Température pendant l’usinage, toujours un peu plus chaude que l’air	Kelvin (environ 310 K)
Rotational speed	Vitesse de rotation de l’outil	tours/minute (rpm)
Torque	Couple : la « force de torsion » que l’outil doit fournir	Newton-mètre (Nm)
Tool wear	Usure de l’outil : depuis combien de temps il travaille	minutes

In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Type                 10000 non-null  str    
 1   Air temperature      10000 non-null  float64
 2   Process temperature  10000 non-null  float64
 3   Rotational speed     10000 non-null  int64  
 4   Torque               10000 non-null  float64
 5   Tool wear            10000 non-null  int64  
 6   Machine failure      10000 non-null  int64  
dtypes: float64(3), int64(3), str(1)
memory usage: 556.8 KB


In [7]:
df.describe().round(2)

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure
count,10000.0,10000.00,10000.00,10000.00,10000.00,10000.00
mean,300.0,310.01,1538.78,39.99,107.95,0.03
std,2.0,1.48,179.28,9.97,63.65,0.18
min,295.3,305.70,1168.00,3.80,0.00,0.00
25%,298.3,308.80,1423.00,33.20,53.00,0.00
50%,300.1,310.10,1503.00,40.10,108.00,0.00
75%,301.5,311.10,1612.00,46.80,162.00,0.00
max,304.5,313.80,2886.00,76.60,253.00,1.00


In [8]:
df["Type"].value_counts()

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

In [9]:
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())
print("\nNombre de doublons :", df.duplicated().sum())

Valeurs manquantes par colonne :
Type                   0
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
Machine failure        0
dtype: int64

Nombre de doublons : 0


La data est bien compléte

In [10]:
counts = df["Machine failure"].value_counts()
props = df["Machine failure"].value_counts(normalize=True) * 100

resume = pd.DataFrame({"Nombre": counts, "Pourcentage (%)": props.round(2)})
resume.index = ["Pas de panne (0)", "Panne (1)"]
print(resume)

                  Nombre  Pourcentage (%)
Pas de panne (0)    9661            96.61
Panne (1)            339             3.39


Module 1.3

In [11]:
import plotly.express as px
import numpy as np

num_cols = ["Air temperature", "Process temperature", "Rotational speed", "Torque", "Tool wear"]
df["Panne"] = df["Machine failure"].map({0: "Non", 1: "Oui"})   # étiquette lisible pour les graphiques

Pour faire les graphique, Il faut utiliser  (histnorm="probability density") puisque la data est désequilibré, donc les cas de pannes, qui sot 339 ne vont pas etre vu devant les cas de pas de panne qui sont 9661

In [12]:
for col in num_cols:
    fig = px.histogram(
        df, x=col, color="Panne", barmode="overlay",
        histnorm="probability density", marginal="box",
        color_discrete_map={"Non": "steelblue", "Oui": "crimson"},
        title=f"Distribution de {col} selon la panne"
    )
    fig.show()

In [13]:
corr = df[num_cols + ["Machine failure"]].corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Matrice de corrélation")
fig.show()

In [14]:
for col in num_cols:
    tranches = pd.qcut(df[col], q=10, duplicates="drop")
    taux = (df.groupby(tranches, observed=True)["Machine failure"].mean() * 100).reset_index()
    taux[col] = taux[col].astype(str)
    fig = px.bar(taux, x=col, y="Machine failure",
                 labels={"Machine failure": "Taux de panne (%)"},
                 title=f"Taux de panne selon {col}")
    fig.add_hline(y=df["Machine failure"].mean() * 100, line_dash="dash",
                  annotation_text="Taux moyen (3,4 %)")
    fig.show()

# Pour la variable catégorielle Type
taux_type = (df.groupby("Type")["Machine failure"].mean() * 100).reset_index()
px.bar(taux_type, x="Type", y="Machine failure",
       labels={"Machine failure": "Taux de panne (%)"},
       title="Taux de panne selon le type de produit").show()

In [15]:
fig = px.scatter(df, x="Rotational speed", y="Torque", color="Panne",
                 color_discrete_map={"Non": "lightgray", "Oui": "crimson"},
                 opacity=0.6, title="Vitesse vs couple : où se produisent les pannes ?")
fig.show()

d'aprés cette figure on observe que lorsque ont a une valeur de torque élevé et une vitesse rotative faible il est fort probable davoir une panne, et dans le cas inverse aussi  

**Module2**

Ajout des variables physiques:
Une machine tombe souvent en panne quand la puissance est trop forte ou trop faible, ou quand la chaleur ne s’évacue pas (écart de température trop petit)
**donc on donne directement au modèle ces informations physiques au lieu de le laisser les deviner.**


In [16]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

ai4i = fetch_ucirepo(id=601)
df = ai4i.data.features.copy()
df["Machine failure"] = ai4i.data.targets["Machine failure"]

# Variables "ingénieur" (feature engineering)
df["Power"] = df["Torque"] * df["Rotational speed"] * 2 * 3.1416 / 60   # puissance en watts
df["Temp diff"] = df["Process temperature"] - df["Air temperature"]      # écart de température

df.head()

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,Power,Temp diff
0,M,298.1,308.6,1551,42.8,0,0,6951.606816,10.5
1,L,298.2,308.7,1408,46.3,3,0,6826.738688,10.5
2,L,298.1,308.5,1498,49.4,5,0,7749.405664,10.4
3,L,298.2,308.6,1433,39.5,7,0,5927.518520,10.4
4,L,298.2,308.7,1408,40.0,9,0,5897.830400,10.5


In [17]:
# ====== PARAMÈTRES (deviendront des menus dans l'application) ======
TARGET = "Machine failure"
FEATURES = ["Type", "Air temperature", "Process temperature", "Rotational speed",
            "Torque", "Tool wear", "Power", "Temp diff"]
TEST_SIZE = 0.2          # 20 % des données pour le test
SCALER = "standard"      # "standard" ou "minmax"
RANDOM_STATE = 42
# ===================================================================

X = df[FEATURES]
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()
print("Variables catégorielles :", cat_cols)
print("Variables numériques    :", num_cols)

Variables catégorielles : ['Type']
Variables numériques    : ['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Power', 'Temp diff']


C:\Users\21694\AppData\Local\Temp\ipykernel_43468\2006110442.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


Maintenant, La séparation des donnés entre Train et Test s'effectue principalement avec **stratify = y** car elle permet de séparer entre les cas de panne et les cas de pas de panne et apres on prend 20% de chaque cas

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print("Train :", X_train.shape, "| taux de panne :", round(y_train.mean() * 100, 2), "%")
print("Test  :", X_test.shape,  "| taux de panne :", round(y_test.mean() * 100, 2), "%")

Train : (8000, 8) | taux de panne : 3.39 %
Test  : (2000, 8) | taux de panne : 3.4 %


remplir les trous, transformer le texte en nombres, mettre tout à la même échell:

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

scaler = StandardScaler() if SCALER == "standard" else MinMaxScaler()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),        # remplace les trous par la médiane
    ("scaler", scaler),                                    # met toutes les variables à la même échelle
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),  # remplace les trous par la valeur la plus fréquente
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

In [20]:
X_train_prep = preprocessor.fit_transform(X_train)   # apprend sur le train ET transforme
X_test_prep = preprocessor.transform(X_test)          # applique seulement, sans réapprendre

feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_prep, columns=feature_names)

print(X_train_df.shape)
X_train_df.describe().round(2)

(8000, 10)


,num__Air temperature,num__Process temperature,num__Rotational speed,num__Torque,num__Tool wear,num__Power,num__Temp diff,cat__Type_H,cat__Type_L,cat__Type_M
count,8000.00,8000.00,8000.00,8000.00,8000.00,8000.00,8000.0,8000.0,8000.00,8000.00
mean,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.0,0.1,0.60,0.30
std,1.00,1.00,1.00,1.00,1.00,1.00,1.0,0.3,0.49,0.46
min,-2.36,-2.91,-2.05,-3.61,-1.69,-4.79,-2.4,0.0,0.00,0.00
25%,-0.85,-0.82,-0.65,-0.68,-0.86,-0.67,-0.7,0.0,0.00,0.00
50%,0.05,0.06,-0.20,0.01,-0.01,-0.01,-0.2,0.0,1.00,0.00
75%,0.75,0.74,0.41,0.68,0.87,0.68,1.0,0.0,1.00,1.00
max,2.25,2.56,7.44,3.65,2.28,3.90,2.1,1.0,1.00,1.00


**Module 3**

In [21]:
!pip install -q xgboost imbalanced-learn


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Pour XGBoost :
ratio = (y_train == 0).sum() / (y_train == 1).sum()

MODELS = {
    "Régression logistique": {
        "model": LogisticRegression(class_weight="balanced", max_iter=2000),
        "params": {"clf__C": [0.01, 0.1, 1, 10]},
    },
    "Arbre de décision": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=42),
        "params": {"clf__max_depth": [3, 5, 8, None],
                   "clf__min_samples_leaf": [1, 5, 20]},
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {"clf__n_neighbors": [5, 11, 21],
                   "clf__weights": ["uniform", "distance"]},
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        "params": {"clf__n_estimators": [200, 400],
                   "clf__max_depth": [None, 10],
                   "clf__min_samples_leaf": [1, 3]},
    },
    "XGBoost": {
        "model": XGBClassifier(scale_pos_weight=ratio, eval_metric="logloss",
                               random_state=42, n_jobs=-1),
        "params": {"clf__n_estimators": [200, 400],
                   "clf__max_depth": [3, 5],
                   "clf__learning_rate": [0.05, 0.1]},
    },
}

# ====== PARAMÈTRE (deviendra des cases à cocher dans l'application) ======
SELECTED_MODELS = list(MODELS.keys())   # ou par exemple ["Random Forest", "XGBoost"]

In [23]:
import time
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_models = {}
cv_results = {}

for name in SELECTED_MODELS:
    cfg = MODELS[name]

    if name == "KNN":
        pipe = ImbPipeline([("prep", preprocessor),
                            ("smote", SMOTE(random_state=42)),
                            ("clf", cfg["model"])])
    else:
        pipe = Pipeline([("prep", preprocessor), ("clf", cfg["model"])])

    search = GridSearchCV(pipe, cfg["params"], scoring="average_precision",
                          cv=cv, n_jobs=-1)

    start = time.time()
    search.fit(X_train, y_train)
    duree = time.time() - start

    best_models[name] = search.best_estimator_
    cv_results[name] = {"PR-AUC (CV)": search.best_score_,
                        "Meilleurs hyperparamètres": search.best_params_,
                        "Temps (s)": duree}
    print(f"{name:22s} | PR-AUC CV = {search.best_score_:.3f} | {duree:.1f} s | {search.best_params_}")

Régression logistique  | PR-AUC CV = 0.493 | 4.5 s | {'clf__C': 0.1}
Arbre de décision      | PR-AUC CV = 0.800 | 0.5 s | {'clf__max_depth': 8, 'clf__min_samples_leaf': 5}
KNN                    | PR-AUC CV = 0.527 | 0.9 s | {'clf__n_neighbors': 21, 'clf__weights': 'distance'}
Random Forest          | PR-AUC CV = 0.868 | 8.4 s | {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 400}
XGBoost                | PR-AUC CV = 0.835 | 1.6 s | {'clf__learning_rate': 0.05, 'clf__max_depth': 5, 'clf__n_estimators': 400}


In [24]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score)

rows = []
for name, model in best_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "Modèle": name,
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Précision (panne)": precision_score(y_test, pred, zero_division=0),
        "Rappel (panne)": recall_score(y_test, pred),
        "F1 (panne)": f1_score(y_test, pred),
        "PR-AUC (CV)": cv_results[name]["PR-AUC (CV)"],
        "Temps (s)": cv_results[name]["Temps (s)"],
    })

comparaison = (pd.DataFrame(rows)
               .sort_values("PR-AUC", ascending=False)
               .reset_index(drop=True)
               .round(3))
comparaison

,Modèle,ROC-AUC,PR-AUC,Précision (panne),Rappel (panne),F1 (panne),PR-AUC (CV),Temps (s)
0,XGBoost,0.973,0.865,0.757,0.824,0.789,0.835,1.623
1,Random Forest,0.968,0.849,0.915,0.794,0.850,0.868,8.404
2,Arbre de décision,0.920,0.793,0.408,0.853,0.552,0.800,0.491
3,KNN,0.931,0.551,0.252,0.824,0.386,0.527,0.877
4,Régression logistique,0.929,0.450,0.171,0.868,0.285,0.493,4.489


**Les Matrices de confusion**

In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

SEUIL = 0.5
noms = list(best_models.keys())

fig = make_subplots(rows=1, cols=len(noms), subplot_titles=noms, horizontal_spacing=0.06)
lignes = []

for i, name in enumerate(noms, start=1):
    proba = best_models[name].predict_proba(X_test)[:, 1]
    pred = (proba >= SEUIL).astype(int)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    texte = [[f"TN<br>{tn}", f"FP<br>{fp}"],
             [f"FN<br>{fn}", f"TP<br>{tp}"]]

    fig.add_trace(go.Heatmap(
        z=cm, x=["Prédit : non", "Prédit : panne"], y=["Réel : non", "Réel : panne"],
        text=texte, texttemplate="%{text}", colorscale="Blues", showscale=False
    ), row=1, col=i)

    lignes.append({"Modèle": name, "TN": tn, "FP (fausses alertes)": fp,
                   "FN (pannes ratées)": fn, "TP (pannes détectées)": tp,
                   "Pannes détectées": f"{tp}/{tp + fn}"})

fig.update_yaxes(autorange="reversed")
fig.update_layout(height=380, width=320 * len(noms),
                  title=f"Matrices de confusion (seuil = {SEUIL})")
fig.show()

pd.DataFrame(lignes)

,Modèle,TN,FP (fausses alertes),FN (pannes ratées),TP (pannes détectées),Pannes détectées
0,Régression logistique,1645,287,9,59,59/68
1,Arbre de décision,1848,84,10,58,58/68
2,KNN,1766,166,12,56,56/68
3,Random Forest,1927,5,14,54,54/68
4,XGBoost,1914,18,12,56,56/68


XGBoost est le meilleur modèle : il détecte 82 % des pannes avec peu de fausses alertes. Random Forest est presque aussi bon mais trop prudent au seuil de 0,5.

**Module 4**

In [26]:
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import (roc_curve, precision_recall_curve, roc_auc_score,
                             average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

probas = {name: model.predict_proba(X_test)[:, 1] for name, model in best_models.items()}

In [27]:
fig = go.Figure()
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                             name=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})"))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                         line=dict(dash="dash", color="gray"), name="Hasard (AUC = 0,5)"))
fig.update_layout(title="Courbes ROC", xaxis_title="Taux de fausses alertes (FPR)",
                  yaxis_title="Taux de pannes détectées (Rappel)", width=750, height=550)
fig.show()

In [28]:
fig = go.Figure()
for name, proba in probas.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines",
                             name=f"{name} (PR-AUC = {average_precision_score(y_test, proba):.3f})"))

base = y_test.mean()
fig.add_hline(y=base, line_dash="dash", line_color="gray",
              annotation_text=f"Hasard ({base:.3f})")
fig.update_layout(title="Courbes précision-rappel", xaxis_title="Rappel (pannes détectées)",
                  yaxis_title="Précision (alertes justes)", width=750, height=550)
fig.show()

In [29]:
import ipywidgets as widgets

def explorer_seuil(modele, seuil):
    proba = probas[modele]
    pred = (proba >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    print(f"Modèle : {modele}   |   Seuil : {seuil:.2f}")
    print(f"Rappel    : {recall_score(y_test, pred):.3f}  → {tp}/{tp + fn} pannes détectées")
    print(f"Précision : {precision_score(y_test, pred, zero_division=0):.3f}  → {fp} fausses alertes")
    print(f"F1        : {f1_score(y_test, pred):.3f}")
    print(f"Pannes ratées (FN) : {fn}")

    # Évolution de la précision, du rappel et du F1 selon le seuil
    seuils = np.linspace(0.01, 0.99, 99)
    precs = [precision_score(y_test, proba >= s, zero_division=0) for s in seuils]
    recs = [recall_score(y_test, proba >= s) for s in seuils]
    f1s = [f1_score(y_test, proba >= s) for s in seuils]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=seuils, y=precs, name="Précision"))
    fig.add_trace(go.Scatter(x=seuils, y=recs, name="Rappel"))
    fig.add_trace(go.Scatter(x=seuils, y=f1s, name="F1", line=dict(dash="dot")))
    fig.add_vline(x=seuil, line_color="red", annotation_text=f"seuil = {seuil:.2f}")
    fig.update_layout(title=f"Effet du seuil – {modele}", xaxis_title="Seuil de décision",
                      yaxis_title="Score", width=750, height=420)
    fig.show()

widgets.interact(
    explorer_seuil,
    modele=widgets.Dropdown(options=list(probas.keys()), value="XGBoost", description="Modèle"),
    seuil=widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="Seuil"),
);

interactive(children=(Dropdown(description='Modèle', index=4, options=('Régression logistique', 'Arbre de déci…

**le meilleur seuil de chaque modèle** : Ce tableau donne, pour chaque modèle, le seuil qui maximise le F1.

In [30]:
lignes = []
for name, proba in probas.items():
    prec, rec, thr = precision_recall_curve(y_test, proba)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    i = np.argmax(f1)
    pred = (proba >= thr[i]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    lignes.append({"Modèle": name, "Meilleur seuil": round(thr[i], 2),
                   "F1": round(f1[i], 3), "Rappel": round(rec[i], 3),
                   "Précision": round(prec[i], 3), "Pannes détectées": f"{tp}/{tp + fn}",
                   "Fausses alertes": fp})

pd.DataFrame(lignes).sort_values("F1", ascending=False).reset_index(drop=True)

,Modèle,Meilleur seuil,F1,Rappel,Précision,Pannes détectées,Fausses alertes
0,Random Forest,0.50,0.859,0.809,0.917,55/68,5
1,XGBoost,0.95,0.836,0.750,0.944,51/68,3
2,Arbre de décision,0.97,0.835,0.779,0.898,53/68,6
3,KNN,0.96,0.595,0.485,0.767,33/68,10
4,Régression logistique,0.79,0.500,0.647,0.407,44/68,64


Même nombre de pannes détectées, mais moins de fausses alertes. Random Forest était juste trop prudent à 0,5 : en baissant son seuil, il devient le meilleur modèle.

In [31]:
import joblib, sklearn

joblib.dump({
    "model": best_models["Random Forest"],
    "threshold": 0.28,
    "features": FEATURES,
    "model_name": "Random Forest",
}, "model.joblib", compress=3)

print("Version scikit-learn :", sklearn.__version__)   # à noter !



Version scikit-learn : 1.9.1


**Module 6**


In [1]:
%%writefile api.py
from typing import Literal
import joblib
import numpy as np
import pandas as pd
import shap
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

bundle = joblib.load("model.joblib")
model = bundle["model"]
THRESHOLD = float(bundle["threshold"])
FEATURES = bundle["features"]
MODEL_NAME = bundle.get("model_name", "Modèle")

prep = model.named_steps["prep"]
clf = model.named_steps["clf"]
explainer = shap.TreeExplainer(clf)
FEATURE_NAMES = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]

LIBELLES = {"Air temperature": "la température de l'air", "Process temperature": "la température du process",
            "Rotational speed": "la vitesse de rotation", "Torque": "le couple",
            "Tool wear": "l'usure de l'outil", "Power": "la puissance", "Temp diff": "l'écart de température",
            "Type_L": "le type de produit L", "Type_M": "le type de produit M", "Type_H": "le type de produit H"}
UNITES = {"Air temperature": "K", "Process temperature": "K", "Rotational speed": "rpm",
          "Torque": "Nm", "Tool wear": "min", "Power": "W", "Temp diff": "K"}

app = FastAPI(title="API Maintenance prédictive – AI4I 2020",
              description="Prédit le risque de panne d'une machine et explique la prédiction avec SHAP.",
              version="1.0")


class Machine(BaseModel):
    type: Literal["L", "M", "H"] = Field(..., description="Qualité du produit : L, M ou H", examples=["L"])
    air_temperature: float = Field(..., gt=0, description="Température de l'air (K)", examples=[302.5])
    process_temperature: float = Field(..., gt=0, description="Température du process (K)", examples=[310.9])
    rotational_speed: float = Field(..., gt=0, description="Vitesse de rotation (rpm)", examples=[1290])
    torque: float = Field(..., ge=0, description="Couple (Nm)", examples=[65.0])
    tool_wear: float = Field(..., ge=0, description="Usure de l'outil (min)", examples=[210])


def to_dataframe(m):
    df = pd.DataFrame([{"Type": m.type, "Air temperature": m.air_temperature,
                        "Process temperature": m.process_temperature,
                        "Rotational speed": m.rotational_speed, "Torque": m.torque, "Tool wear": m.tool_wear}])
    df["Power"] = df["Torque"] * df["Rotational speed"] * 2 * 3.1416 / 60
    df["Temp diff"] = df["Process temperature"] - df["Air temperature"]
    return df[FEATURES]


def valeur_lisible(f, df):
    if f.startswith("Type_"):
        return str(df["Type"].iloc[0])
    return f"{df[f].iloc[0]:.1f} {UNITES.get(f, '')}".strip()


@app.get("/", summary="Vérifier que l'API fonctionne")
def accueil():
    return {"statut": "ok", "modele": MODEL_NAME, "seuil": THRESHOLD, "variables": FEATURES}


@app.post("/predict", summary="Probabilité de panne et décision")
def predict(machine: Machine):
    proba = float(model.predict_proba(to_dataframe(machine))[0, 1])
    return {"probabilite_panne": round(proba, 4), "seuil": THRESHOLD,
            "decision": "Panne probable" if proba >= THRESHOLD else "Fonctionnement normal"}


@app.post("/explain", summary="Explication SHAP pour une machine")
def explain(machine: Machine, top_k: int = 5):
    if top_k < 1:
        raise HTTPException(status_code=400, detail="top_k doit être >= 1")
    df = to_dataframe(machine)
    proba = float(model.predict_proba(df)[0, 1])
    exp = explainer(prep.transform(df), check_additivity=False)
    values, base = np.asarray(exp.values[0]), np.asarray(exp.base_values[0])
    if values.ndim == 2:
        values, base = values[:, 1], np.ravel(base)[1]
    base = float(np.ravel(base)[0])

    contributions = sorted(
        ({"variable": f, "valeur": valeur_lisible(f, df), "contribution": round(float(v), 4),
          "effet": "augmente le risque" if v > 0 else "diminue le risque"}
         for f, v in zip(FEATURE_NAMES, values)),
        key=lambda c: abs(c["contribution"]), reverse=True)

    if proba >= THRESHOLD:
        causes = [c for c in contributions if c["contribution"] > 0][:2]
        debut, lien = "Le risque de panne est élevé", "principalement à cause de"
    else:
        causes = [c for c in contributions if c["contribution"] < 0][:2]
        debut, lien = "Le risque de panne est faible", "principalement grâce à"
    details = " et ".join(f"{LIBELLES.get(c['variable'], c['variable'])} ({c['valeur']})" for c in causes)
    synthese = f"{debut} ({proba:.0%}), {lien} {details}." if causes else f"{debut} ({proba:.0%})."

    return {"probabilite_panne": round(proba, 4),
            "decision": "Panne probable" if proba >= THRESHOLD else "Fonctionnement normal",
            "valeur_de_base": round(base, 4), "contributions": contributions[:top_k],
            "synthese": synthese}

Writing api.py


In [2]:
import requests

API_URL = "http://127.0.0.1:8000"

# 1. Vérifier que l'API répond
print(requests.get(f"{API_URL}/").json())

{'statut': 'ok', 'modele': 'Random Forest', 'seuil': 0.28, 'variables': ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Power', 'Temp diff']}


In [3]:
# 2. Une machine à risque (couple élevé, vitesse faible, outil usé)
machine = {"type": "L", "air_temperature": 302.5, "process_temperature": 310.9,
           "rotational_speed": 1290, "torque": 65.0, "tool_wear": 210}

print(requests.post(f"{API_URL}/predict", json=machine).json())

{'probabilite_panne': 0.9525, 'seuil': 0.28, 'decision': 'Panne probable'}


In [6]:
# 3. L'explication SHAP et la phrase de synthèse
import pandas as pd

reponse = requests.post(f"{API_URL}/explain", json=machine).json()
print(reponse["synthese"])
pd.DataFrame(reponse["contributions"])
reponse = requests.post(f"{API_URL}/explain", json=machine).json()

print(reponse["synthese"])
pd.DataFrame(reponse["contributions"])

Le risque de panne est élevé (95%), principalement à cause de le couple (65.0 Nm) et la vitesse de rotation (1290.0 rpm).
Le risque de panne est élevé (95%), principalement à cause de le couple (65.0 Nm) et la vitesse de rotation (1290.0 rpm).


,variable,valeur,contribution,effet
0,Torque,65.0 Nm,0.1212,augmente le risque
1,Rotational speed,1290.0 rpm,0.1110,augmente le risque
2,Tool wear,210.0 min,0.1016,augmente le risque
3,Temp diff,8.4 K,0.0617,augmente le risque
4,Power,8780.8 W,0.0254,augmente le risque


In [7]:
# 4. Une machine normale, pour comparer
machine_normale = {"type": "M", "air_temperature": 300.0, "process_temperature": 310.0,
                   "rotational_speed": 1500, "torque": 40.0, "tool_wear": 20}

print(requests.post(f"{API_URL}/explain", json=machine_normale).json()["synthese"])

Le risque de panne est faible (0%), principalement grâce à la puissance (6283.2 W) et le couple (40.0 Nm).


**Interface Streamlit**

In [8]:
import os
os.makedirs("pages", exist_ok=True)

In [9]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Maintenance prédictive explicable", page_icon="⚙️", layout="wide")

st.title("⚙️ Maintenance prédictive explicable")
st.subheader("Cas 15 – AI4I 2020 : prédire et expliquer les pannes de machines")

st.markdown("""
Cette application prédit si une machine va tomber en panne **et explique pourquoi**.

**Suis les modules dans l'ordre (menu à gauche) :**
1. **Données** : téléchargement automatique, statistiques, visualisations
2. **Préparation** : variables, encodage, normalisation, séparation train/test
3. **Modélisation** : 5 modèles optimisés et comparés
4. **Évaluation** : courbes ROC, précision-rappel, matrice de confusion, seuil
5. **Explainable AI** : SHAP et LIME
6. **API** : interrogation du service FastAPI
""")

st.info("Les données sont déséquilibrées (environ 3,4 % de pannes) : on évalue les modèles avec "
        "le rappel, le F1 et la PR-AUC, pas avec l'accuracy.")

Writing app.py


In [10]:
%%writefile pages/1_Donnees.py
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Module 1 – Données", page_icon="📊", layout="wide")
st.title("📊 Module 1 – Données")
COULEURS = {"Non": "#4C78A8", "Oui": "#E45756"}


@st.cache_data(show_spinner="Téléchargement de AI4I 2020 depuis l'UCI…")
def charger_ai4i():
    from ucimlrepo import fetch_ucirepo
    d = fetch_ucirepo(id=601)
    df = d.data.features.copy()
    df["Machine failure"] = d.data.targets["Machine failure"].values
    return df


# ---------- Chargement ----------
source = st.radio("Source des données",
                  ["AI4I 2020 (téléchargement automatique UCI)", "Importer un fichier CSV"], horizontal=True)

if source.startswith("AI4I"):
    df = charger_ai4i()
    target = "Machine failure"
    st.success("Jeu AI4I 2020 téléchargé automatiquement (UCI, id = 601). Les colonnes TWF, HDF, PWF, "
               "OSF, RNF sont écartées : elles révèlent la panne (fuite de données).")
else:
    fichier = st.file_uploader("Fichier CSV", type="csv")
    if fichier is None:
        st.info("Importe un fichier CSV pour continuer.")
        st.stop()
    df = pd.read_csv(fichier, sep=None, engine="python")
    target = st.selectbox("Variable cible", df.columns, index=len(df.columns) - 1)

valeurs = sorted(df[target].dropna().unique().tolist(), key=str)
if len(valeurs) < 2:
    st.error("La cible doit contenir au moins deux valeurs.")
    st.stop()
if set(valeurs) != {0, 1}:
    pos = st.selectbox("Classe positive (l'événement à prédire)", valeurs, index=len(valeurs) - 1)
    df = df.copy()
    df[target] = (df[target] == pos).astype(int)

# Mémoriser les données pour les autres modules
st.session_state["df"] = df
st.session_state["target"] = target
y = df[target]

# ---------- Chiffres clés ----------
c1, c2, c3, c4 = st.columns(4)
c1.metric("Lignes", len(df))
c2.metric("Colonnes", df.shape[1])
c3.metric("Valeurs manquantes", int(df.isnull().sum().sum()))
c4.metric("Classe positive", f"{y.mean():.1%}")

t1, t2, t3, t4, t5 = st.tabs(["Aperçu", "Statistiques", "Valeurs manquantes",
                              "Proportion des classes", "Visualisations"])

with t1:
    st.dataframe(df.head(50))
    st.dataframe(pd.DataFrame({"Type": df.dtypes.astype(str), "Valeurs distinctes": df.nunique()}))

with t2:
    st.dataframe(df.describe().T.round(2))
    cat = df.select_dtypes(exclude="number").columns
    if len(cat):
        st.dataframe(df[cat].describe().T)

with t3:
    st.dataframe(pd.DataFrame({"Manquantes": df.isnull().sum(), "%": (df.isnull().mean() * 100).round(2)}))
    st.write(f"Doublons : **{int(df.duplicated().sum())}**")

with t4:
    prop = pd.DataFrame({"Classe": ["Pas de panne (0)", "Panne (1)"],
                         "Nombre": [int((y == 0).sum()), int((y == 1).sum())]})
    prop["%"] = (prop["Nombre"] / len(y) * 100).round(2)
    a, b = st.columns(2)
    a.dataframe(prop, hide_index=True)
    b.plotly_chart(px.pie(prop, names="Classe", values="Nombre", hole=0.5,
                          color_discrete_sequence=[COULEURS["Non"], COULEURS["Oui"]]))
    if y.mean() < 0.2:
        st.warning("Classes déséquilibrées : l'accuracy serait trompeuse (un modèle qui dit toujours "
                   f"« pas de panne » aurait {1 - y.mean():.1%} d'accuracy sans détecter aucune panne).")

with t5:
    d = df.copy()
    d["Panne"] = np.where(y == 1, "Oui", "Non")
    num = [c for c in d.select_dtypes(include="number").columns if c != target]
    cat = [c for c in d.select_dtypes(exclude="number").columns if c != "Panne"]

    st.subheader("Distributions selon la classe")
    st.caption("histnorm = densité : chaque classe est ramenée à sa propre taille, sinon les pannes "
               "seraient invisibles.")
    col = st.selectbox("Variable", num, key="dist")
    st.plotly_chart(px.histogram(d, x=col, color="Panne", barmode="overlay", marginal="box",
                                 histnorm="probability density", color_discrete_map=COULEURS))

    st.subheader("Matrice de corrélation")
    corr = d[num + [target]].corr().round(2)
    st.plotly_chart(px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
                              zmin=-1, zmax=1, height=550))

    st.subheader("Taux de panne par variable")
    var = st.selectbox("Variable", num + cat, key="taux")
    if var in num:
        groupes = pd.qcut(d[var], q=10, duplicates="drop")
        taux = d.groupby(groupes, observed=True)[target].mean().reset_index()
        taux[var] = taux[var].astype(str)
    else:
        taux = d.groupby(var)[target].mean().reset_index()
    taux[target] *= 100
    fig = px.bar(taux, x=var, y=target, labels={target: "Taux de panne (%)"})
    fig.add_hline(y=y.mean() * 100, line_dash="dash", annotation_text=f"Taux moyen ({y.mean():.1%})")
    st.plotly_chart(fig)

    st.subheader("Nuage de points")
    c1, c2 = st.columns(2)
    vx = c1.selectbox("Axe X", num, index=num.index("Rotational speed") if "Rotational speed" in num else 0)
    vy = c2.selectbox("Axe Y", num, index=num.index("Torque") if "Torque" in num else min(1, len(num) - 1))
    st.plotly_chart(px.scatter(d.sort_values("Panne"), x=vx, y=vy, color="Panne", opacity=0.6,
                               color_discrete_map={"Non": "lightgray", "Oui": COULEURS["Oui"]}))

Writing pages/1_Donnees.py


In [13]:
%%writefile pages/2_Preparation.py
import pandas as pd
import streamlit as st
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

st.set_page_config(page_title="Module 2 – Préparation", page_icon="🧹", layout="wide")
st.title("🧹 Module 2 – Préparation")

if "df" not in st.session_state:
    st.info("Ouvre d'abord la page **Donnees** (Module 1) pour charger les données.")
    st.stop()

df = st.session_state["df"].copy()
target = st.session_state["target"]
st.write(f"Variable cible : **{target}**  (1 = panne, 0 = pas de panne)")

# ---------- Variables physiques (AI4I uniquement) ----------
ai4i = {"Torque", "Rotational speed", "Air temperature", "Process temperature"} <= set(df.columns)
physique = False
if ai4i:
    physique = st.checkbox("Ajouter les variables physiques : Power (couple × vitesse) "
                           "et Temp diff (écart de température)", value=True)
    if physique:
        df["Power"] = df["Torque"] * df["Rotational speed"] * 2 * 3.1416 / 60
        df["Temp diff"] = df["Process temperature"] - df["Air temperature"]

# Colonnes exclues par défaut : fuite de données et identifiants
EXCLUES = ["TWF", "HDF", "PWF", "OSF", "RNF", "UDI", "Product ID"]
candidats = [c for c in df.columns if c != target]
defaut = [c for c in candidats if c not in EXCLUES
          and not (df[c].dtype == object and df[c].nunique() > 0.5 * len(df))]

# ---------- Paramètres ----------
with st.form("preparation"):
    features = st.multiselect("Variables explicatives", candidats, default=defaut,
                              help="Les identifiants et les colonnes qui révèlent la cible sont exclus par défaut.")
    c1, c2, c3 = st.columns(3)
    num_impute = c1.selectbox("Valeurs manquantes (numériques)", ["median", "mean"],
                              format_func={"median": "Médiane", "mean": "Moyenne"}.get)
    c1.caption("Catégorielles : valeur la plus fréquente.")
    scaler_nom = c2.selectbox("Normalisation", ["StandardScaler", "MinMaxScaler", "Aucune"])
    c2.caption("Encodage des catégories : one-hot.")
    test_size = c3.slider("Part du jeu de test", 0.10, 0.40, 0.20, 0.05)
    c4, c5 = st.columns(2)
    stratify = c4.checkbox("Séparation stratifiée (même proportion de pannes dans train et test)", value=True)
    seed = c5.number_input("random_state", value=42, step=1)
    valider = st.form_submit_button("Appliquer la préparation", type="primary")

if valider:
    if not features:
        st.error("Choisis au moins une variable.")
        st.stop()

    X, y = df[features], df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y if stratify else None, random_state=int(seed))

    cat_cols = X.select_dtypes(exclude="number").columns.tolist()
    num_cols = X.select_dtypes(include="number").columns.tolist()

    scaler = {"StandardScaler": StandardScaler(), "MinMaxScaler": MinMaxScaler(),
              "Aucune": "passthrough"}[scaler_nom]
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy=num_impute)), ("scaler", scaler)])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    preprocessor = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

    # Mémoriser pour les modules suivants
    st.session_state["split"] = {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
                                 "features": features, "num_cols": num_cols, "cat_cols": cat_cols,
                                 "physique": physique}
    st.session_state["preprocessor"] = preprocessor   # non entraîné : il sera ajusté dans chaque pipeline
    for cle in ["results", "probas", "thresholds"]:   # les anciens modèles ne sont plus valables
        st.session_state.pop(cle, None)

# ---------- Résultat ----------
if "split" not in st.session_state:
    st.info("Règle les paramètres puis clique sur **Appliquer la préparation**.")
    st.stop()

sp = st.session_state["split"]
st.success("Préparation appliquée.")
c1, c2, c3 = st.columns(3)
c1.metric("Entraînement", f"{len(sp['X_train'])} lignes", f"{sp['y_train'].mean():.2%} de pannes", delta_color="off")
c2.metric("Test", f"{len(sp['X_test'])} lignes", f"{sp['y_test'].mean():.2%} de pannes", delta_color="off")
c3.metric("Variables", f"{len(sp['num_cols'])} num. + {len(sp['cat_cols'])} cat.")

from sklearn.base import clone
prep = clone(st.session_state["preprocessor"])
Xt = prep.fit_transform(sp["X_train"])          # appris sur le train uniquement
noms = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
apercu = pd.DataFrame(Xt, columns=noms)

st.write("**Train après prétraitement** (imputation, one-hot, normalisation) :")
st.dataframe(apercu.head(10).round(3))
st.dataframe(apercu.describe().T[["mean", "std", "min", "max"]].round(2))
st.caption("Le prétraitement est appris sur le train seulement (fit_transform), puis appliqué tel quel "
           "au test (transform) : le test ne doit rien « souffler » au modèle.")

Overwriting pages/2_Preparation.py


In [14]:
%%writefile pages/3_Modelisation.py
import time

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

st.set_page_config(page_title="Module 3 – Modélisation", page_icon="🤖", layout="wide")
st.title("🤖 Module 3 – Modélisation et comparaison")

if "split" not in st.session_state:
    st.info("Applique d'abord la préparation (page **Preparation**).")
    st.stop()

sp = st.session_state["split"]
X_train, X_test, y_train, y_test = sp["X_train"], sp["X_test"], sp["y_train"], sp["y_test"]

# ---------- Grilles d'hyperparamètres : (valeurs possibles, valeurs cochées par défaut) ----------
GRILLES = {
    "Régression logistique": {"C": ([0.01, 0.1, 1, 10], [0.1, 1])},
    "Arbre de décision": {"max_depth": ([3, 5, 8, None], [5, 8]),
                          "min_samples_leaf": ([1, 5, 20], [1, 5])},
    "KNN": {"n_neighbors": ([5, 11, 21, 31], [11, 21]),
            "weights": (["uniform", "distance"], ["distance"])},
    "Random Forest": {"n_estimators": ([100, 200, 400], [200]),
                      "max_depth": ([None, 10, 20], [None, 10]),
                      "min_samples_leaf": ([1, 3, 5], [1])},
    "XGBoost": {"n_estimators": ([100, 200, 400], [200, 400]),
                "max_depth": ([3, 5, 7], [3, 5]),
                "learning_rate": ([0.03, 0.05, 0.1, 0.2], [0.05, 0.1])},
}
modeles = list(GRILLES) if XGBClassifier else [m for m in GRILLES if m != "XGBoost"]
if XGBClassifier is None:
    st.warning("XGBoost n'est pas installé (pip install xgboost) : il est retiré de la liste.")


def creer_modele(nom, ponderation, ratio):
    cw = "balanced" if ponderation else None
    if nom == "Régression logistique":
        return LogisticRegression(class_weight=cw, max_iter=2000)
    if nom == "Arbre de décision":
        return DecisionTreeClassifier(class_weight=cw, random_state=42)
    if nom == "KNN":
        return KNeighborsClassifier()
    if nom == "Random Forest":
        return RandomForestClassifier(class_weight=cw, random_state=42, n_jobs=-1)
    return XGBClassifier(scale_pos_weight=ratio if ponderation else 1.0,
                         eval_metric="logloss", random_state=42, n_jobs=-1)


# ---------- Paramètres ----------
choix = st.multiselect("Modèles à entraîner", modeles, default=modeles)
c1, c2 = st.columns(2)
deseq = c1.radio("Gestion du déséquilibre",
                 ["Pondération des classes (class_weight / scale_pos_weight)",
                  "SMOTE pour tous les modèles", "Aucune"])
c1.caption("Avec la pondération, KNN (qui n'a pas ce paramètre) utilise SMOTE.")
folds = c2.slider("Validation croisée : nombre de plis", 3, 10, 5)
c2.caption("Critère d'optimisation : PR-AUC, adaptée aux classes déséquilibrées.")

st.subheader("Hyperparamètres testés (GridSearch)")
grilles = {}
for nom in choix:
    with st.expander(nom):
        grilles[nom] = {}
        for param, (options, defaut) in GRILLES[nom].items():
            vals = st.multiselect(param, options, default=defaut, key=f"{nom}_{param}",
                                  format_func=lambda v: "illimitée" if v is None else str(v))
            grilles[nom][f"clf__{param}"] = vals or [defaut[0]]
        nb = int(np.prod([len(v) for v in grilles[nom].values()]))
        st.caption(f"{nb} combinaison(s) × {folds} plis = {nb * folds} entraînements")

# ---------- Entraînement ----------
if st.button("Lancer l'entraînement", type="primary", disabled=not choix):
    ratio = (y_train == 0).sum() / (y_train == 1).sum()
    ponderation = deseq.startswith("Pondération")
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    results, probas = {}, {}
    barre = st.progress(0.0)

    for i, nom in enumerate(choix):
        barre.progress(i / len(choix), text=f"Entraînement : {nom}…")
        prep = clone(st.session_state["preprocessor"])
        clf = creer_modele(nom, ponderation, ratio)
        smote = deseq.startswith("SMOTE") or (nom == "KNN" and ponderation)
        if smote:   # SMOTE appliqué uniquement aux données d'entraînement de chaque pli
            pipe = ImbPipeline([("prep", prep), ("smote", SMOTE(random_state=42)), ("clf", clf)])
        else:
            pipe = Pipeline([("prep", prep), ("clf", clf)])

        search = GridSearchCV(pipe, grilles[nom], scoring="average_precision", cv=cv, n_jobs=-1)
        t0 = time.time()
        search.fit(X_train, y_train)
        results[nom] = {"model": search.best_estimator_, "cv": search.best_score_,
                        "params": {k.replace("clf__", ""): v for k, v in search.best_params_.items()},
                        "temps": time.time() - t0, "smote": smote}
        probas[nom] = search.best_estimator_.predict_proba(X_test)[:, 1]

    barre.progress(1.0, text="Terminé")
    st.session_state["results"] = results
    st.session_state["probas"] = probas
    st.session_state["thresholds"] = {}

if "results" not in st.session_state:
    st.info("Choisis les modèles puis clique sur **Lancer l'entraînement**.")
    st.stop()

# ---------- Tableau comparatif ----------
results, probas = st.session_state["results"], st.session_state["probas"]
lignes, matrices = [], []
for nom, r in results.items():
    p = probas[nom]
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    lignes.append({"Modèle": nom, "PR-AUC": average_precision_score(y_test, p),
                   "ROC-AUC": roc_auc_score(y_test, p),
                   "Précision": precision_score(y_test, pred, zero_division=0),
                   "Rappel": recall_score(y_test, pred), "F1": f1_score(y_test, pred),
                   "PR-AUC (CV)": r["cv"], "Temps (s)": r["temps"],
                   "Meilleurs hyperparamètres": str(r["params"])})
    matrices.append({"Modèle": nom, "TN": tn, "FP (fausses alertes)": fp,
                     "FN (pannes ratées)": fn, "TP (pannes détectées)": tp,
                     "Pannes détectées": f"{tp}/{tp + fn}"})

tab = pd.DataFrame(lignes).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
scores = ["PR-AUC", "ROC-AUC", "Précision", "Rappel", "F1", "PR-AUC (CV)"]

st.subheader("Tableau comparatif (jeu de test, seuil 0,5)")
st.dataframe(tab.style.format({**{c: "{:.3f}" for c in scores}, "Temps (s)": "{:.1f}"})
             .background_gradient(cmap="Greens", subset=["PR-AUC", "Rappel", "F1"]), hide_index=True)

st.plotly_chart(px.bar(tab.melt(id_vars="Modèle", value_vars=["PR-AUC", "ROC-AUC", "F1", "Rappel"]),
                       x="Modèle", y="value", color="variable", barmode="group",
                       labels={"value": "Score", "variable": "Métrique"}))

st.subheader("Détail des erreurs (seuil 0,5)")
st.dataframe(pd.DataFrame(matrices), hide_index=True)

meilleur = tab.iloc[0]
st.info(f"Meilleur modèle selon la PR-AUC : **{meilleur['Modèle']}** ({meilleur['PR-AUC']:.3f}). "
        "Si la PR-AUC en validation croisée et celle du test sont proches, le modèle généralise bien "
        "(pas de surapprentissage). Le seuil de 0,5 n'est pas forcément le bon : on l'ajuste au Module 4.")

Writing pages/3_Modelisation.py


In [15]:
%%writefile pages/4_Evaluation.py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)

st.set_page_config(page_title="Module 4 – Évaluation", page_icon="📈", layout="wide")
st.title("📈 Module 4 – Évaluation")

if "results" not in st.session_state:
    st.info("Entraîne d'abord des modèles (page **Modelisation**).")
    st.stop()

y_test = st.session_state["split"]["y_test"]
probas = st.session_state["probas"]
thresholds = st.session_state.setdefault("thresholds", {})

# ---------- Courbes ROC et précision-rappel ----------
c1, c2 = st.columns(2)

fig = go.Figure()
for nom, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{nom} ({roc_auc_score(y_test, p):.3f})"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="gray"),
                         name="Hasard (0,5)"))
fig.update_layout(title="Courbes ROC", xaxis_title="Taux de fausses alertes (FPR)",
                  yaxis_title="Rappel (pannes détectées)", height=460)
c1.plotly_chart(fig)

fig = go.Figure()
for nom, p in probas.items():
    prec, rec, _ = precision_recall_curve(y_test, p)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines",
                             name=f"{nom} ({average_precision_score(y_test, p):.3f})"))
fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"Hasard ({y_test.mean():.3f})")
fig.update_layout(title="Courbes précision-rappel", xaxis_title="Rappel (pannes détectées)",
                  yaxis_title="Précision (alertes justes)", height=460)
c2.plotly_chart(fig)

st.caption("Avec des classes déséquilibrées, les courbes ROC se ressemblent toutes et paraissent excellentes ; "
           "la courbe précision-rappel révèle les vraies différences entre modèles.")

# ---------- Curseur de seuil ----------
st.subheader("Curseur de seuil de décision")
st.caption("Le modèle donne une probabilité de panne ; au-dessus du seuil, il déclenche une alerte. "
           "Seuil bas : plus de pannes détectées mais plus de fausses alertes. Seuil haut : l'inverse.")

noms = list(probas)
meilleur = max(noms, key=lambda n: average_precision_score(y_test, probas[n]))
nom = st.selectbox("Modèle", noms, index=noms.index(meilleur))
seuil = st.slider("Seuil", 0.01, 0.99, float(thresholds.get(nom, 0.5)), 0.01, key=f"seuil_{nom}")
thresholds[nom] = seuil   # mémorisé pour les modules 5 et 6

p = probas[nom]
pred = (p >= seuil).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

k = st.columns(5)
k[0].metric("Rappel", f"{recall_score(y_test, pred, zero_division=0):.3f}", f"{tp}/{tp + fn} détectées",
            delta_color="off")
k[1].metric("Précision", f"{precision_score(y_test, pred, zero_division=0):.3f}")
k[2].metric("F1", f"{f1_score(y_test, pred, zero_division=0):.3f}")
k[3].metric("Fausses alertes (FP)", int(fp))
k[4].metric("Pannes ratées (FN)", int(fn))

c1, c2 = st.columns([1, 1.4])

texte = [[f"TN<br>{tn}", f"FP<br>{fp}"], [f"FN<br>{fn}", f"TP<br>{tp}"]]
fig = go.Figure(go.Heatmap(z=[[tn, fp], [fn, tp]], x=["Prédit : non", "Prédit : panne"],
                           y=["Réel : non", "Réel : panne"], text=texte, texttemplate="%{text}",
                           colorscale="Blues", showscale=False))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title=f"Matrice de confusion – {nom} (seuil {seuil:.2f})", height=380)
c1.plotly_chart(fig)

# Précision, rappel et F1 pour tous les seuils (calcul vectorisé, instantané)
seuils = np.linspace(0.01, 0.99, 99)
yv = y_test.to_numpy()
P = p[None, :] >= seuils[:, None]
TP = (P & (yv == 1)).sum(axis=1)
FP = (P & (yv == 0)).sum(axis=1)
FN = (yv == 1).sum() - TP
prec = np.divide(TP, TP + FP, out=np.zeros(len(seuils)), where=(TP + FP) > 0)
rec = TP / (TP + FN)
f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(len(seuils)), where=(prec + rec) > 0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=seuils, y=prec, name="Précision"))
fig.add_trace(go.Scatter(x=seuils, y=rec, name="Rappel"))
fig.add_trace(go.Scatter(x=seuils, y=f1, name="F1", line=dict(dash="dot")))
fig.add_vline(x=seuil, line_color="red", annotation_text=f"seuil = {seuil:.2f}")
fig.update_layout(title="Effet du seuil", xaxis_title="Seuil", yaxis_title="Score", height=380)
c2.plotly_chart(fig)

# ---------- Meilleur seuil de chaque modèle ----------
st.subheader("Seuil qui maximise le F1 pour chaque modèle")
lignes = []
for n, pp in probas.items():
    pr, rc, th = precision_recall_curve(y_test, pp)
    f = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-9)
    i = int(np.argmax(f))
    pr_ = (pp >= th[i]).astype(int)
    a, b, c, d = confusion_matrix(y_test, pr_, labels=[0, 1]).ravel()
    lignes.append({"Modèle": n, "Seuil optimal": round(float(th[i]), 2), "F1": f[i], "Rappel": rc[i],
                   "Précision": pr[i], "Pannes détectées": f"{d}/{d + c}", "Fausses alertes": int(b)})
tab = pd.DataFrame(lignes).sort_values("F1", ascending=False).reset_index(drop=True)
st.dataframe(tab.style.format({"F1": "{:.3f}", "Rappel": "{:.3f}", "Précision": "{:.3f}"}), hide_index=True)

if st.button("Appliquer ces seuils optimaux à tous les modèles"):
    for _, r in tab.iterrows():
        thresholds[r["Modèle"]] = float(r["Seuil optimal"])
        st.session_state.pop(f"seuil_{r['Modèle']}", None)   # remet le curseur sur la nouvelle valeur
    st.rerun()

st.caption("Limite : le seuil est choisi sur le jeu de test, ces chiffres sont donc légèrement optimistes. "
           "En maintenance, on peut préférer un seuil plus bas si une panne ratée coûte plus cher qu'une "
           "inspection inutile : c'est une décision métier.")

Writing pages/4_Evaluation.py


In [16]:
%%writefile pages/6_API.py
import os

import pandas as pd
import plotly.express as px
import requests
import streamlit as st

st.set_page_config(page_title="Module 6 – API", page_icon="🔌", layout="wide")
st.title("🔌 Module 6 – API d'interprétation (FastAPI)")

st.markdown("L'application envoie les caractéristiques d'une machine au service **FastAPI** (`api.py`), "
            "qui renvoie la probabilité de panne, la décision et l'explication SHAP. "
            "Lancement de l'API dans un autre terminal : `uvicorn api:app --reload`")

url = st.text_input("Adresse de l'API", os.environ.get("API_URL", "http://127.0.0.1:8000")).rstrip("/")
st.markdown(f"Documentation interactive (Swagger) : [{url}/docs]({url}/docs)")

# ---------- État de l'API ----------
try:
    info = requests.get(f"{url}/", timeout=5).json()
    st.success(f"API connectée – modèle : **{info['modele']}**, seuil de décision : **{info['seuil']}**")
except Exception:
    st.error("API injoignable. Vérifie qu'elle tourne (`uvicorn api:app --reload`) et l'adresse ci-dessus.")
    st.stop()

# ---------- Formulaire ----------
st.subheader("Caractéristiques de la machine")
with st.form("machine"):
    c = st.columns(3)
    machine = {
        "type": c[0].selectbox("Type de produit", ["L", "M", "H"]),
        "air_temperature": c[1].number_input("Température de l'air (K)", 290.0, 310.0, 302.5, 0.1),
        "process_temperature": c[2].number_input("Température du process (K)", 300.0, 320.0, 310.9, 0.1),
        "rotational_speed": c[0].number_input("Vitesse de rotation (rpm)", 1000.0, 3000.0, 1290.0, 10.0),
        "torque": c[1].number_input("Couple (Nm)", 0.0, 90.0, 65.0, 0.5),
        "tool_wear": c[2].number_input("Usure de l'outil (min)", 0.0, 260.0, 210.0, 1.0),
    }
    top_k = st.slider("Nombre de variables dans l'explication", 1, 10, 5)
    envoyer = st.form_submit_button("Envoyer à l'API", type="primary")

if not envoyer:
    st.info("Les valeurs par défaut décrivent une machine à risque (couple élevé, vitesse faible, outil usé). "
            "Essaie aussi une machine normale : couple 40, vitesse 1500, usure 20.")
    st.stop()

# ---------- Appel de l'API ----------
try:
    r = requests.post(f"{url}/explain", params={"top_k": top_k}, json=machine, timeout=60)
    r.raise_for_status()
    res = r.json()
except Exception as e:
    st.error(f"Échec de l'appel à l'API : {e}")
    st.stop()

a, b = st.columns(2)
a.metric("Probabilité de panne", f"{res['probabilite_panne']:.1%}")
b.metric("Décision", res["decision"])

if res["decision"] == "Panne probable":
    st.error(res["synthese"])
else:
    st.success(res["synthese"])

dc = pd.DataFrame(res["contributions"]).iloc[::-1]
dc["Variable"] = dc["variable"] + " = " + dc["valeur"]
fig = px.bar(dc, x="contribution", y="Variable", orientation="h", color="effet",
             color_discrete_map={"augmente le risque": "#E45756", "diminue le risque": "#4C78A8"},
             labels={"contribution": "Contribution SHAP"},
             title="Pourquoi le modèle a pris cette décision")
st.plotly_chart(fig)
st.caption("Barre rouge : la variable pousse vers la panne. Barre bleue : elle rassure le modèle.")

with st.expander("Réponse JSON brute renvoyée par l'API"):
    st.json(res)

Writing pages/6_API.py


In [17]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Maintenance prédictive explicable", page_icon="⚙️", layout="wide")

st.title("⚙️ Maintenance prédictive explicable")
st.subheader("Cas 15 – AI4I 2020 : prédire et expliquer les pannes de machines")

st.markdown("""
Cette application prédit si une machine va tomber en panne **et explique pourquoi**.

**Suis les modules dans l'ordre (menu à gauche) :**
1. **Données** : téléchargement automatique, statistiques, visualisations
2. **Préparation** : variables, encodage, normalisation, séparation train/test
3. **Modélisation** : 5 modèles optimisés et comparés
4. **Évaluation** : courbes ROC, précision-rappel, matrice de confusion, seuil
6. **API** : interrogation du service FastAPI, avec explication SHAP de la prédiction
""")

st.info("Les données sont déséquilibrées (environ 3,4 % de pannes) : on évalue les modèles avec "
        "le rappel, le F1 et la PR-AUC, pas avec l'accuracy.")

Overwriting app.py


In [18]:
%%writefile .gitignore
.venv/
__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store

Writing .gitignore


In [19]:
%%writefile requirements.txt
pandas
numpy
scikit-learn
xgboost
imbalanced-learn
ucimlrepo
plotly
shap
joblib
fastapi
uvicorn
requests
streamlit
ipykernel
ipywidgets
nbformat

Overwriting requirements.txt


In [20]:
import os
os.makedirs("captures", exist_ok=True)

In [24]:
%%writefile README.md
# ⚙️ Maintenance prédictive explicable – AI4I 2020

**Projet Challenge 1 – Application web de classification explicable**
Data Mining · ENSI · 2026-2027 · Enseignante : Rym Besrour

Cas métier n°15 : **prédire les pannes de machines industrielles** et expliquer la décision du modèle.

## 🔗 Liens
- Application web : *(lien à ajouter après le déploiement)*
- Documentation de l'API : `http://127.0.0.1:8000/docs` (en local)

## 📊 Données
- Jeu **AI4I 2020 Predictive Maintenance** (UCI, id = 601), téléchargé automatiquement avec `ucimlrepo`.
- 10 000 machines, 6 variables : type de produit (L/M/H), température de l'air, température du process,
  vitesse de rotation, couple, usure de l'outil.
- Cible : `Machine failure` (1 = panne). **Classes déséquilibrées : 3,4 % de pannes.**
- Colonnes écartées : TWF, HDF, PWF, OSF, RNF (types de panne, fuite de données), UDI et Product ID (identifiants).
- Variables ajoutées : **Power** (couple × vitesse) et **Temp diff** (écart de température), liées aux
  mécanismes physiques de panne.

## 🧠 Démarche
| Module | Contenu |
|---|---|
| 1. Données | Téléchargement automatique ou import CSV, statistiques, valeurs manquantes, visualisations |
| 2. Préparation | Choix des variables, imputation, one-hot, normalisation, séparation train/test stratifiée |
| 3. Modélisation | Régression logistique, arbre, KNN, Random Forest, XGBoost, GridSearch (critère PR-AUC) |
| 4. Évaluation | ROC, précision-rappel, matrice de confusion, curseur de seuil en direct |
| 6. API | Service FastAPI (prédiction + explication SHAP), appelé par l'application |

**Gestion du déséquilibre :** séparation stratifiée, `class_weight="balanced"` / `scale_pos_weight`,
SMOTE pour KNN, évaluation par rappel, F1 et PR-AUC (et non l'accuracy).

## 🏆 Résultats (jeu de test, 68 pannes sur 2 000 machines)
| Modèle | PR-AUC | ROC-AUC | Pannes détectées (seuil 0,5) | Fausses alertes |
|---|---|---|---|---|
| XGBoost | 0,865 | 0,973 | 56/68 | 18 |
| Random Forest | 0,859 | 0,970 | 48/68 | 3 |
| Arbre de décision | 0,793 | 0,920 | 58/68 | 84 |
| KNN | 0,551 | 0,931 | 56/68 | 166 |
| Régression logistique | 0,450 | 0,929 | 59/68 | 287 |

**Modèle retenu : Random Forest avec un seuil de 0,28** → 56/68 pannes détectées (rappel 0,82)
avec seulement **5 fausses alertes** (précision 0,92).

Le problème est non linéaire (les pannes viennent de combinaisons comme couple élevé + vitesse faible),
ce qui explique l'avance des modèles à base d'arbres sur la régression logistique.

## 🚀 Lancer le projet en local
```bash
python -m venv .venv
.venv\Scripts\activate          # Windows  (Mac/Linux : source .venv/bin/activate)
pip install -r requirements.txt
```
Terminal 1 – l'API :
```bash
uvicorn api:app --reload
```
Terminal 2 – l'application :
```bash
streamlit run app.py
```

## 🔌 API
| Endpoint | Rôle |
|---|---|
| `GET /` | État de l'API, modèle et seuil chargés |
| `POST /predict` | Probabilité de panne et décision |
| `POST /explain` | Contributions SHAP et phrase de synthèse |

Exemple de réponse : *« Le risque de panne est élevé (95 %), principalement à cause du couple (65,0 Nm)
et de la vitesse de rotation (1290,0 rpm). »*

## 📁 Structure
├── app.py # page d’accueil Streamlit
├── pages/ # un fichier par module
├── api.py # API FastAPI
├── model.joblib # modèle Random Forest + seuil
├── Projet_Challenge_1.ipynb
├── requirements.txt
└── captures/
## 📸 Captures d'écran
![Données](captures/donnees.png)
![Préparation](captures/preparation.png)
![Modélisation](captures/modelisation.png)
![Évaluation](captures/evaluation.png)
![API dans l'application](captures/api_app.png)
![Documentation Swagger](captures/swagger.png)

## ⚠️ Limites
- Le seuil optimal a été choisi sur le jeu de test : résultats légèrement optimistes.
- AI4I 2020 est un jeu de données synthétique, inspiré de conditions industrielles réelles.


Overwriting README.md
